# Customer Segmentation Using K-Means Clustering

This notebook performs customer segmentation on the **Online Shoppers Intention** dataset using the K-Means clustering algorithm.  
The goal is to group customers into meaningful segments based on their browsing behavior,  
specifically their `ProductRelated_Duration` and `BounceRates`.

---

## Table of Contents
1. Imports
2. Dataset Loading
3. Data Exploration
4. Data Cleaning
5. Feature Selection
6. Feature Scaling
7. Elbow Method
8. Silhouette Score
9. K-Means Training
10. Cluster Visualization
11. Cluster Analysis
12. Evaluation
13. Conclusion

---
## 1. Imports

---
## 2. Dataset Loading

In [ ]:
# Use a relative path so the notebook is portable
DATA_PATH = os.path.join(os.getcwd(), "online_shoppers_intention.csv")

data = pd.read_csv(DATA_PATH)
print(f"Dataset loaded successfully: {data.shape[0]} rows, {data.shape[1]} columns")

---
## 3. Data Exploration

In [ ]:
# Preview the first few rows
data.head()

In [ ]:
# Column data types and non-null counts
data.info()

In [ ]:
# Descriptive statistics for numeric columns
data.describe()

In [ ]:
# Check the shape and column names
print(f"Shape: {data.shape}")
print(f"\nColumns: {list(data.columns)}")

---
## 4. Data Cleaning

Check for missing values and verify data types.

In [ ]:
# Check for null values in each column
null_counts = data.isnull().sum()
print("Null values per column:")
print(null_counts[null_counts > 0] if null_counts.any() else "No missing values found.")

In [ ]:
# Check for duplicate rows
n_duplicates = data.duplicated().sum()
print(f"Number of duplicate rows: {n_duplicates}")

# Drop duplicates if any
if n_duplicates > 0:
    data = data.drop_duplicates().reset_index(drop=True)
    print(f"After removing duplicates: {data.shape[0]} rows remain.")

In [ ]:
# Verify data types — ensure numeric columns are truly numeric
print("Data types:")
print(data.dtypes)

---
## 5. Feature Selection

For this analysis we focus on two behavioral features:
- **ProductRelated_Duration**: Time spent on product-related pages  
- **BounceRates**: Percentage of visitors who leave after viewing only one page

These features capture browsing intensity and engagement.

In [ ]:
# Define the features used for clustering
FEATURE_COLS = ["ProductRelated_Duration", "BounceRates"]

X = data[FEATURE_COLS].values
print(f"Feature matrix shape: {X.shape}")
print(f"Features used: {FEATURE_COLS}")

---
## 6. Feature Scaling

K-Means is distance-based, so features must be on the same scale.  
We apply **StandardScaler** (zero mean, unit variance) to avoid one feature dominating the distance calculation.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Scaled feature matrix shape: {X_scaled.shape}")
print(f"Mean after scaling:  {X_scaled.mean(axis=0).round(6)}")
print(f"Std after scaling:   {X_scaled.std(axis=0).round(6)}")

---
## 7. Elbow Method

The Elbow Method plots **WCSS** (Within-Cluster Sum of Squares / Inertia)  
against the number of clusters *k* to help identify the optimal *k*.

In [ ]:
# Range of k values to evaluate
K_RANGE = range(2, 11)

wcss = []
for k in K_RANGE:
    km = KMeans(
        n_clusters=k,
        init="k-means++",
        max_iter=300,
        n_init=10,
        random_state=RANDOM_STATE,
    )
    km.fit(X_scaled)
    wcss.append(km.inertia_)

# Plot the Elbow curve
plt.figure(figsize=(10, 6))
plt.plot(list(K_RANGE), wcss, marker="o", linewidth=2)
plt.xlabel("Number of Clusters (k)", fontsize=12)
plt.ylabel("WCSS (Inertia)", fontsize=12)
plt.title("Elbow Method — Optimal k Selection", fontsize=14)
plt.xticks(list(K_RANGE))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 8. Silhouette Score

The Silhouette Score measures how similar each point is to its own cluster compared  
to other clusters.  Higher scores indicate better-defined clusters.

We compute it for each candidate *k* and select the one with the **highest** score.

In [ ]:
silhouette_scores = []
for k in K_RANGE:
    km = KMeans(
        n_clusters=k,
        init="k-means++",
        max_iter=300,
        n_init=10,
        random_state=RANDOM_STATE,
    )
    labels = km.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)
    print(f"k = {k:>2d}  |  Silhouette Score = {score:.4f}")

# Identify the best k based on Silhouette Score
best_k = list(K_RANGE)[np.argmax(silhouette_scores)]
print(f"\n>>> Best k by Silhouette Score: {best_k}")

# Plot
plt.figure(figsize=(10, 6))
plt.plot(list(K_RANGE), silhouette_scores, marker="s", linewidth=2, color="green")
plt.xlabel("Number of Clusters (k)", fontsize=12)
plt.ylabel("Silhouette Score", fontsize=12)
plt.title("Silhouette Score vs. Number of Clusters", fontsize=14)
plt.xticks(list(K_RANGE))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 9. K-Means Training

Based on the Elbow Method and Silhouette Score analysis,  
we select the optimal number of clusters and train the final model.

In [ ]:
# Use the best k determined above (you can also override manually)
OPTIMAL_K = best_k
print(f"Training K-Means with k = {OPTIMAL_K}")

kmeans_model = KMeans(
    n_clusters=OPTIMAL_K,
    init="k-means++",
    max_iter=300,
    n_init=10,
    random_state=RANDOM_STATE,
)

# Fit and predict cluster labels
cluster_labels = kmeans_model.fit_predict(X_scaled)

# Attach cluster labels to the original DataFrame
data["Cluster"] = cluster_labels

print(f"Cluster label distribution:\n{pd.Series(cluster_labels).value_counts().sort_index()}")

---
## 10. Cluster Visualization

Scatter plot of the two selected features, colour-coded by cluster assignment,  
with cluster centroids overlaid.

In [ ]:
# Use a colour palette with enough distinct colours
palette = sns.color_palette("tab10", n_colors=OPTIMAL_K)

plt.figure(figsize=(12, 8))

# Plot each cluster
for i in range(OPTIMAL_K):
    mask = cluster_labels == i
    plt.scatter(
        X_scaled[mask, 0],
        X_scaled[mask, 1],
        label=f"Cluster {i}",
        color=palette[i],
        alpha=0.6,
        edgecolors="w",
        linewidth=0.3,
        s=30,
    )

# Plot centroids
centroids = kmeans_model.cluster_centers_
plt.scatter(
    centroids[:, 0],
    centroids[:, 1],
    marker="X",
    s=250,
    c="black",
    edgecolors="yellow",
    linewidths=2,
    label="Centroids",
    zorder=5,
)

plt.xlabel("ProductRelated_Duration (scaled)", fontsize=12)
plt.ylabel("BounceRates (scaled)", fontsize=12)
plt.title(f"K-Means Clustering (k={OPTIMAL_K})", fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 11. Cluster Analysis

Examine the characteristics of each cluster using the **original** (unscaled) feature values.

In [ ]:
# Summary statistics per cluster (original scale)
cluster_summary = data.groupby("Cluster")[FEATURE_COLS].agg(
    ["mean", "median", "std", "min", "max", "count"]
)
print("Cluster summary (original feature scale):")
cluster_summary

In [ ]:
# Revenue rate per cluster (if 'Revenue' column exists)
if "Revenue" in data.columns:
    revenue_rate = (
        data.groupby("Cluster")["Revenue"]
        .mean()
        .reset_index()
        .rename(columns={"Revenue": "Revenue_Rate"})
    )
    print("\nRevenue rate (proportion of purchasers) per cluster:")
    print(revenue_rate)

---
## 12. Evaluation

### 12.1 Adjusted Rand Index (ARI)
We compare the cluster labels against the **Revenue** column  
(treated as ground-truth binary labels) using the Adjusted Rand Index.  
ARI ranges from -1 to 1 where 1 is perfect agreement and 0 is random.

### 12.2 Confusion Matrix

In [ ]:
if "Revenue" in data.columns:
    # Encode ground truth labels
    le = LabelEncoder()
    labels_true = le.fit_transform(data["Revenue"])
    labels_pred = cluster_labels

    # Adjusted Rand Index
    ari = adjusted_rand_score(labels_true, labels_pred)
    print(f"Adjusted Rand Index (ARI): {ari:.4f}")

    # Silhouette Score of the final model
    final_sil = silhouette_score(X_scaled, cluster_labels)
    print(f"Final Silhouette Score:     {final_sil:.4f}")

    # Confusion matrix (only meaningful when OPTIMAL_K == 2)
    if OPTIMAL_K == 2:
        cm = confusion_matrix(labels_true, labels_pred)
        plt.figure(figsize=(6, 5))
        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=[f"Cluster {i}" for i in range(OPTIMAL_K)],
            yticklabels=le.classes_,
        )
        plt.xlabel("Predicted Cluster")
        plt.ylabel("True Label (Revenue)")
        plt.title("Confusion Matrix: Clusters vs Revenue")
        plt.tight_layout()
        plt.show()
    else:
        print(f"\nNote: Confusion matrix is skipped because k={OPTIMAL_K} != 2.")
else:
    print("'Revenue' column not found — skipping supervised evaluation.")

---
## 13. Conclusion

### Key Findings

1. **Optimal Clusters**: The Elbow Method and Silhouette Score analysis were used  
   together to determine the optimal number of customer segments.

2. **Feature Scaling**: Applying `StandardScaler` before K-Means is essential  
   because the raw features (`ProductRelated_Duration` and `BounceRates`) operate  
   on vastly different scales.

3. **Cluster Interpretation**:
   - Customers were segmented based on browsing duration and bounce behaviour.
   - Clusters with higher `ProductRelated_Duration` and lower `BounceRates`  
     tend to correlate with higher purchase (Revenue) rates.

4. **Evaluation**: The Adjusted Rand Index gives a quantitative measure of how well  
   the unsupervised clusters align with the known Revenue labels.

### Limitations & Future Work

- Only two features were used; including more (e.g., `PageValues`, `ExitRates`)  
  may yield richer segments.
- Consider experimenting with other clustering algorithms such as DBSCAN or  
  Gaussian Mixture Models.
- Dimensionality reduction (PCA, t-SNE) can help visualise higher-dimensional  
  cluster spaces.